<a id="kmeans-objective"></a>

$$
\text{K 均值聚类 (K-Means Clustering): 无监督学习, 把 n 个点分成 K 组} \\ \quad \\
\text{目标函数: 簇内平方和 (Within-Cluster Sum of Squares, WCSS, sklearn 里叫 inertia)} \\
J(C, \mu) = \sum_{k=1}^K \sum_{i \in C_k} \| x_i - \mu_k \|^2 \\
C_k \text{: 第 k 个簇包含的样本, } \mu_k \text{: 第 k 个簇的中心} \\ \quad \\
\text{同时优化 "分组" 和 "中心" 很难 (分组的方式有 } K^n \text{ 量级种, NP-hard)} \\
\text{办法: 交替优化 (Alternating Optimization), 固定一个, 优化另一个}
$$

<a id="lloyd"></a>

$$
\text{Lloyd 算法: 两个步骤都可以精确求解} \\ \quad \\
\text{1. 分配步 (Assignment): 固定中心 } \mu \text{, 优化分组} \\
J = \sum_{i=1}^n \| x_i - \mu_{c(i)} \|^2 \text{ (} c(i) \text{ 是样本 i 所属的簇), 每一项只和 } c(i) \text{ 有关, 可以分开最小化} \\
c(i) = \arg\min_k \| x_i - \mu_k \|^2 \quad (\text{每个点分给最近的中心}) \\ \quad \\
\text{2. 更新步 (Update): 固定分组, 优化中心. 每个簇独立, 对 } \mu_k \text{ 求梯度令其为 0:} \\
\frac{\partial}{\partial \mu_k} \sum_{i \in C_k} \| x_i - \mu_k \|^2 = \sum_{i \in C_k} -2 (x_i - \mu_k) = 0 \implies |C_k| \mu_k = \sum_{i \in C_k} x_i \implies \mu_k = \frac{1}{|C_k|} \sum_{i \in C_k} x_i \\
\text{Hessian = } 2 |C_k| I \text{ 正定, 所以均值就是最小值点 (这也是 "K 均值" 名字的由来)} \\ \quad \\
\text{另一种证明 (不用求导), 记 } \bar{x} \text{ 为簇内均值, 交叉项 } \sum_i (x_i - \bar{x}) = 0 \text{:} \\
\sum_{i} \| x_i - \mu \|^2 = \sum_{i} \| (x_i - \bar{x}) + (\bar{x} - \mu) \|^2 = \sum_{i} \| x_i - \bar{x} \|^2 + |C_k| \, \| \bar{x} - \mu \|^2 \geq \sum_{i} \| x_i - \bar{x} \|^2 \\ \quad \\
\text{收敛性: 每一步都是在固定另一组变量时的最优解, 所以 J 单调不增; } J \geq 0 \text{ 且分组方式有限} \\
\implies \text{有限步内一定停止. 但只保证局部最优 (local optimum), 结果依赖初始中心}
$$

数学基础: [偏导数](../math_basics/calculus.ipynb#partial) · [极值条件](../math_basics/calculus.ipynb#extremum) · [方差的分解 (和贝塞尔校正证明里的技巧一样)](../math_basics/statistics.ipynb#bessel)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

X, y_true = make_blobs(n_samples=400, centers=[[-4, 0], [0, 4], [3, -1], [5, 5]], cluster_std=1.1, random_state=0)

def wcss(X, labels, centers):
    return np.sum((X - centers[labels])**2)

def assign(X, centers):
    dist2 = ((X[:, None, :] - centers[None, :, :])**2).sum(axis=2)   # n x K 的距离平方矩阵
    return np.argmin(dist2, axis=1)

def lloyd(X, init_centers, max_iter=100):
    centers = init_centers.copy()
    history = []
    for it in range(max_iter):
        labels = assign(X, centers)
        history.append(wcss(X, labels, centers))          # 分配步之后
        new_centers = np.array([X[labels == k].mean(axis=0) for k in range(len(centers))])
        history.append(wcss(X, labels, new_centers))      # 更新步之后
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels, centers, np.array(history), it + 1

# 均值是簇内平方和的最小值点: 和随便取的点比较
pts = X[y_true == 0]
mean_pt = pts.mean(axis=0)
other = mean_pt + np.array([0.3, -0.2])
print(f'sum |x - mean|^2 = {np.sum((pts - mean_pt)**2):.4f}, sum |x - other|^2 = {np.sum((pts - other)**2):.4f}, '
      f'差 = {np.sum((pts - other)**2) - np.sum((pts - mean_pt)**2):.4f}, n |mean - other|^2 = {len(pts) * np.sum((mean_pt - other)**2):.4f}')

rng = np.random.default_rng(3)
init = X[rng.choice(len(X), 4, replace=False)]
labels, centers, history, n_iter = lloyd(X, init)
print(f'迭代 {n_iter} 次收敛, J 单调不增? {np.all(np.diff(history) <= 1e-9)}')
print(f'J 的变化 (每次迭代后) = {np.round(history[1::2], 2)}')

sk = KMeans(n_clusters=4, init=init, n_init=1, algorithm='lloyd', tol=0, max_iter=300).fit(X)
print(f'和 sklearn (相同初始中心) 比较: 中心一致? {np.allclose(centers, sk.cluster_centers_)}, 标签一致? {np.array_equal(labels, sk.labels_)}, '
      f'inertia: 手算 = {wcss(X, labels, centers):.6f}, sklearn = {sk.inertia_:.6f}')

<a id="kmeanspp"></a>

$$
\text{K-means++ 初始化: 让初始中心尽量分散, 避免坏的局部最优} \\
1. \ \text{第一个中心从样本中均匀随机选} \\
2. \ \text{计算每个点到 "已选中心中最近那个" 的距离 } D(x) \\
3. \ \text{以概率 } \frac{D(x)^2}{\sum_{x'} D(x')^2} \text{ 选下一个中心 (离现有中心越远, 越可能被选中)} \\
4. \ \text{重复 2, 3 直到选出 K 个} \\
\text{(理论保证: 期望的 J 不超过最优值的 } O(\ln K) \text{ 倍)} \\
\text{它只是降低陷入坏局部最优的概率, 所以实际中会从不同初始值运行多次 (sklearn 的 n\_init), 取 J 最小的结果}
$$

In [ ]:
def kmeans_pp_init(X, K, rng):
    centers = [X[rng.integers(len(X))]]
    for _ in range(K - 1):
        D2 = np.min(((X[:, None, :] - np.array(centers)[None, :, :])**2).sum(axis=2), axis=1)
        prob = D2 / D2.sum()
        centers.append(X[rng.choice(len(X), p=prob)])
    return np.array(centers)

# 比较 100 次: 随机初始化 vs k-means++ (用 9 个簇的网格数据, 簇越多随机初始化越容易把两个中心放进同一个簇)
X_grid, _ = make_blobs(n_samples=450, centers=[[i * 4, j * 4] for i in range(3) for j in range(3)], cluster_std=0.6, random_state=1)
rng = np.random.default_rng(7)
J_random, J_pp = [], []
for _ in range(100):
    J_random.append(lloyd(X_grid, X_grid[rng.choice(len(X_grid), 9, replace=False)])[2][-1])
    J_pp.append(lloyd(X_grid, kmeans_pp_init(X_grid, 9, rng))[2][-1])
J_random, J_pp = np.array(J_random), np.array(J_pp)
J_best = min(J_random.min(), J_pp.min())
print(f'随机初始化: 平均 J = {J_random.mean():.2f}, 陷入坏局部最优 (J > 1.01 * 最优) 的比例 = {np.mean(J_random > 1.01 * J_best):.2f}')
print(f'k-means++ : 平均 J = {J_pp.mean():.2f}, 陷入坏局部最优的比例 = {np.mean(J_pp > 1.01 * J_best):.2f}')

worst = X_grid[np.random.default_rng(0).choice(len(X_grid), 9, replace=False)]
lab_w, cen_w, hist_w, _ = lloyd(X_grid, worst)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].scatter(X_grid[:, 0], X_grid[:, 1], c=lab_w, s=6, cmap='tab10')
axs[0].scatter(cen_w[:, 0], cen_w[:, 1], c='k', marker='x', s=80, label='centers (one random init)')
axs[0].set_title(f'J = {hist_w[-1]:.1f}, best J = {J_best:.1f}')
axs[1].hist(J_random / J_best, bins=30, alpha=0.6, label='random init')
axs[1].hist(J_pp / J_best, bins=30, alpha=0.6, label='k-means++')
axs[1].set_xlabel('final J / best J')
plot_nicely()
plt.show()

<a id="elbow"></a>

$$
\text{选 K: 肘部法则 (Elbow Method)} \\
\text{K 越大 J 一定越小 (K = n 时每个点自成一簇, J = 0), 所以不能直接最小化 J} \\
\text{画出 J 随 K 的变化, 找下降速度突然变慢的 "肘部": 再增加簇数带来的收益很小} \\
\text{(其他方法: 轮廓系数 Silhouette, 信息准则, 或者根据业务需求决定)}
$$

In [ ]:
Ks = range(1, 10)
J_by_K = []
rng = np.random.default_rng(11)
for K in Ks:
    best_J = np.inf
    for _ in range(10):                               # 每个 K 用 10 次 k-means++ 取最好的
        lab, cen, hist, _ = lloyd(X, kmeans_pp_init(X, K, rng))
        best_J = min(best_J, hist[-1])
    J_by_K.append(best_J)
J_by_K = np.array(J_by_K)
sk_J = [KMeans(n_clusters=K, n_init=10, random_state=0).fit(X).inertia_ for K in Ks]
print(f'J(K) 手算  = {np.round(J_by_K, 1)}')
print(f'J(K) sklearn = {np.round(sk_J, 1)}')
print(f'相对下降 (J(K-1) - J(K)) / J(K-1) = {np.round(-np.diff(J_by_K) / J_by_K[:-1], 3)} -> K = 4 之后下降很小')

plt.figure(figsize=(4, 3))
plt.plot(list(Ks), J_by_K, 'o-', label='WCSS (manual)')
plt.plot(list(Ks), sk_J, 'x--', label='inertia (sklearn)')
plt.axvline(4, color='r', linestyle=':', label='elbow K = 4')
plt.xlabel('K')
plot_nicely()
plt.show()